# 당뇨 예측 - XGBoost Optuna SHAP 분석

- 타겟: `당뇨유병` (0: 없음 / 1: 있음)
- 데이터: x1_preprocessed.csv
- 모델: XGBoost (Optuna 최적 파라미터)
- 분석: OOF 전체 기준 SHAP 피처 기여도 해석
- Threshold: 0.45 고정
- 검증: Stratified 5-Fold CV

In [ ]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
matplotlib.rcParams["font.family"] = "DejaVu Sans"


# ── 경로 설정 ──────────────────────────────────────────────
INPUT_PATH = "/Users/Jiyeon/Desktop/final_project/ML/data/x1_preprocessed.csv"
NPY_DIR = "/Users/Jiyeon/Desktop/final_project/ML/outputs/oof"
RANDOM_STATE = 42
THRESHOLD = 0.45

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"로드 완료 | shape: {df.shape}")

## 2. 피처 / 타겟 분리

In [ ]:
TARGET = "당뇨유병"
DROP_COLS = ["고혈압유병", "당뇨유병", "이상지질혈증유병", "비만단계"]

data = df.dropna(subset=[TARGET]).copy()
X = data.drop(columns=DROP_COLS)
y = data[TARGET].astype(int)
neg, pos = (y == 0).sum(), (y == 1).sum()
ratio = neg / pos
print(f"샘플 수: {len(y)}  |  정상: {neg}  |  당뇨: {pos}")

## 3. Optuna 최적 파라미터 설정

In [ ]:
best_params = {
    "n_estimators": 449,
    "learning_rate": 0.08341579758246154,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.9784695865033908,
    "colsample_bytree": 0.8903503412662817,
    "gamma": 0.8602388794499964,
    "reg_alpha": 0.36719351888544843,
    "reg_lambda": 2.3212709626465675,
    "scale_pos_weight": ratio,
    "eval_metric": "auc",
    "early_stopping_rounds": 50,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}
print("파라미터 설정 완료")

## 4. Stratified 5-Fold CV — OOF proba & SHAP 수집

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(y))
shap_values = np.zeros(X.shape)
fold_scores = []

print("=" * 55)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**best_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    proba = model.predict_proba(X_val)[:, 1]
    oof_proba[val_idx] = proba

    explainer = shap.TreeExplainer(model)
    shap_values[val_idx] = explainer.shap_values(X_val)

    pred = (proba >= THRESHOLD).astype(int)
    fold_scores.append(
        {
            "fold": fold,
            "auc": roc_auc_score(y_val, proba),
            "f1": f1_score(y_val, pred),
            "recall": recall_score(y_val, pred),
        }
    )
    print(
        f"  Fold {fold} | AUC: {fold_scores[-1]['auc']:.4f} | "
        f"Recall: {fold_scores[-1]['recall']:.4f} | F1: {fold_scores[-1]['f1']:.4f}"
    )

scores_df = pd.DataFrame(fold_scores)
print("=" * 55)
print(
    f"  평균   | AUC: {scores_df.auc.mean():.4f} | "
    f"Recall: {scores_df.recall.mean():.4f} | F1: {scores_df.f1.mean():.4f}"
)

## OOF proba 저장 (.npy)

In [ ]:
os.makedirs(NPY_DIR, exist_ok=True)
npy_path = os.path.join(NPY_DIR, "oof_proba_DM_xgboost_optuna_shap.npy")
oof_array = np.stack([oof_proba, y.values], axis=1)  # col0: proba, col1: y_true
np.save(npy_path, oof_array)
print(f"저장 완료 → {npy_path}")
print(f"shape: {oof_array.shape}  (col0: oof_proba, col1: y_true)")
loaded = np.load(npy_path)
print(f"로드 확인: shape={loaded.shape}, 일치={np.allclose(oof_array, loaded)}")

---
# SHAP 분석

## 5. SHAP Feature Importance (mean |SHAP|)

In [ ]:
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
mean_shap.head(20)[::-1].plot(kind="barh", color="darkorange")
plt.title("SHAP Feature Importance Top 20 (mean |SHAP|) — 당뇨 XGBoost Optuna · OOF 전체")
plt.xlabel("mean |SHAP value|")
plt.tight_layout()
plt.show()

print("[SHAP Feature Importance Top 15]")
for i, (feat, val) in enumerate(mean_shap.head(15).items(), 1):
    print(f"  {i:2d}. {feat}: {val:.4f}")

## 6. SHAP Summary Plot (Beeswarm)

In [ ]:
shap.summary_plot(shap_values, X, plot_type="dot", max_display=20, show=False)
plt.title("SHAP Summary (Beeswarm) — 당뇨 XGBoost Optuna · OOF 전체", fontsize=12)
plt.tight_layout()
plt.show()

## 7. SHAP Dependence Plot — 상위 피처 (나이, BMI)

In [ ]:
top2 = mean_shap.head(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, feat in zip(axes, top2):
    shap.dependence_plot(feat, shap_values, X, ax=ax, show=False)
    ax.set_title(f"SHAP Dependence — {feat} (Optuna)")
plt.tight_layout()
plt.show()

## 8. SHAP Dependence Plot — 당뇨 가족력 3종

In [ ]:
fh_feats = ["당뇨가족력_형제", "당뇨가족력_모", "당뇨가족력_부"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, feat in zip(axes, fh_feats):
    shap.dependence_plot(feat, shap_values, X, ax=ax, show=False)
    ax.set_title(f"SHAP Dependence — {feat}")
plt.suptitle("당뇨 가족력 3종 SHAP Dependence", fontsize=12)
plt.tight_layout()
plt.show()

## 9. SHAP Waterfall — 고위험 / 저위험 샘플 개별 해석

In [ ]:
explainer_last = shap.TreeExplainer(model)

high_idx = np.argsort(oof_proba)[::-1][0]
low_idx = np.argsort(oof_proba)[0]

for idx, label in [(high_idx, "고위험"), (low_idx, "저위험")]:
    shap_exp = explainer_last(X.iloc[[idx]])
    print(f"[{label}] 샘플 index={idx} | proba={oof_proba[idx]:.4f} | 실제={y.iloc[idx]}")
    shap.waterfall_plot(shap_exp[0], show=False)
    plt.title(f"SHAP Waterfall — {label} 샘플 (proba={oof_proba[idx]:.3f})")
    plt.tight_layout()
    plt.show()

## 10. Baseline gain vs Optuna SHAP 비교

In [ ]:
# Baseline gain importance (threshold 조정 단계 결과)
baseline_gain = {
    "나이": 33.7,
    "당뇨가족력_모": 16.1,
    "당뇨가족력_부": 15.9,
    "당뇨가족력_형제": 15.5,
    "BMI": 12.6,
    "성별": 11.8,
    "고지혈증가족력_형제": 11.3,
    "직업_농림어업": 9.8,
    "고지혈증가족력_모": 9.2,
    "직업_관리전문": 9.2,
    "음주빈도_enc": 9.0,
    "체중": 8.9,
    "직업_사무": 7.8,
    "고혈압가족력_모": 7.7,
    "직업_서비스판매": 7.4,
}
# gain 정규화 (0~1)
max_gain = max(baseline_gain.values())
baseline_norm = {k: v / max_gain for k, v in baseline_gain.items()}

# SHAP 정규화
max_shap = mean_shap.max()
shap_norm = (mean_shap / max_shap).head(15).to_dict()

# 공통 피처만 비교
common = sorted(set(baseline_norm.keys()) & set(shap_norm.keys()), key=lambda x: shap_norm.get(x, 0), reverse=True)

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(len(common))
w = 0.35
ax.barh(x - w / 2, [baseline_norm[f] for f in common], w, label="Baseline (gain, 정규화)", color="#6b7599", alpha=0.8)
ax.barh(
    x + w / 2,
    [shap_norm[f] for f in common],
    w,
    label="Optuna SHAP (mean|shap|, 정규화)",
    color="darkorange",
    alpha=0.9,
)
ax.set_yticks(x)
ax.set_yticklabels(common)
ax.set_xlabel("Normalized Importance")
ax.set_title("Feature Importance 비교 — Baseline gain vs Optuna SHAP")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. DB 로그 저장

In [ ]:
import sys

sys.path.insert(0, "/Users/Jiyeon/Desktop/final_project/ML")
from model_logger import ModelLogger

logger = ModelLogger("/Users/Jiyeon/Desktop/final_project/ML/model_result.db")

pred_oof = (oof_proba >= THRESHOLD).astype(int)
cm = confusion_matrix(y, pred_oof)

run_id = logger.log_run(
    target_var="당뇨",
    model_name="XGBoost",
    stage="optuna_shap",
    hyperparams={
        "learning_rate": best_params["learning_rate"],
        "max_depth": best_params["max_depth"],
        "n_estimators": best_params["n_estimators"],
        "class_weight": {0: 1.0, 1: round(ratio, 4)},
        "min_child_weight": best_params["min_child_weight"],
        "subsample": best_params["subsample"],
        "colsample_bytree": best_params["colsample_bytree"],
        "gamma": best_params["gamma"],
        "reg_alpha": best_params["reg_alpha"],
        "reg_lambda": best_params["reg_lambda"],
    },
    data_info={
        "feature_count": X.shape[1],
        "train_test_split": "5-Fold CV",
        "scaling_method": "None",
    },
    oof_metrics={
        "accuracy": float((pred_oof == y).mean()),
        "recall": recall_score(y, pred_oof),
        "precision": precision_score(y, pred_oof),
        "f1_score": f1_score(y, pred_oof),
        "auc_roc": roc_auc_score(y, oof_proba),
        "cm": cm.tolist(),
    },
    fold_scores=scores_df.to_dict("records"),
    top_features=mean_shap.head(15).to_dict(),
    note="Optuna 최적 파라미터 기준 SHAP 분석. top_features = SHAP mean|shap| 기준.",
)

print(f"저장 완료 → run_id: {run_id}")
print()
print("[전체 실험 목록]")
print(logger.compare_runs().to_string(index=False))